# Anomaly check

QC on satellite GVF vs PhenoCam GCC and NDVI: scores table, gap by veg boxplots.

**Spin-up** (`gvf_sos == 1`): the phenology fit failed and landed on DOY 1 by
accident, not because green-up really started on Jan 1. On flat, low amplitude
curves (EN, sparse shrub, evergreen) there is no clear winter to summer swing, so
it pin SOS at the first day of data. That inflates gap /
divergence vs NDVI or GCC (noise misread as signal), so spin-up sites must be
flagged and usually excluded before interpreting lag or compression.

Artifacts: `anomaly_pipeline/output/` (`metadata/` scores, `boxplot/`,
`golden_standard_ranking.csv`).

**Veg Codes** DB = deciduous broadleaf, EN = evergreen needle, GR = grassland, AG = agriculture, SH = shrub


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "anomaly_pipeline":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared.data_collection import (
    build_golden_ranking,
    collect_folder,
    group_summary,
    load_all_scores,
    load_table,
    plot_gap_boxplot_by_veg,
    top_n,
)

# GVF text in plotting stage (drop once, reuse here)
INPUT_DIR = REPO / "plotting_pipeline" / "input"
ANOMALY_DIR = REPO / "anomaly_pipeline" / "output"
METADATA_DIR = ANOMALY_DIR / "metadata"


## Score all folders

One row per site-year: SOS/MOS/DOS/EOS for GVF, GCC, and NDVI, plus pairwise
gap / DTW / divergence. Use it to find spin-up (`gvf_sos == 1`), large land-type
offsets, and other bad fits without opening every plot.

Scores every available input folder under `plotting_pipeline/input/` and writes
`anomaly_pipeline/output/metadata/<FOLDER>_scores.csv`. Missing folders are skipped.


In [ ]:
FOLDERS = [
    "GBOV_2023",
    "GBOV_2024",
    "GoldenSites_2023",
    "GoldenSites_2024",
]
LIMIT = None
SORT = "gvf_vs_ndvi_div"
TOP = 10

score_paths = {}
for folder in FOLDERS:
    src = INPUT_DIR / folder
    if not src.is_dir():
        print(f"skip {folder}: no folder at {src}")
        continue
    csv_path = collect_folder(folder, INPUT_DIR, ANOMALY_DIR, limit=LIMIT)
    score_paths[folder] = csv_path
    df = load_table(csv_path)
    spin = int(df["gvf_sos"].eq(1.0).sum()) if "gvf_sos" in df.columns else 0
    print(f"{folder}: {len(df)} rows | spin-up={spin} | {csv_path.name}")


In [ ]:
# peek: top rows + veg summary for each scored folder
for folder, csv_path in score_paths.items():
    df = load_table(csv_path)
    cols = [
        "site", "lag", "greenup_comp", "senescence_comp", "veg", "year",
        "gvf_sos", "gcc_sos", "ndvi_sos",
        "gvf_vs_ndvi_div", "gvf_vs_ndvi_gap", "gvf_vs_ndvi_dtw",
        "gvf_vs_gcc_div", "gcc_vs_ndvi_div",
    ]
    cols = [c for c in cols if c in df.columns]
    print(f"\n=== {folder} ===")
    display(top_n(df, by=SORT, n=TOP)[cols])
    display(group_summary(df, by="veg"))


## Satellite Data Gap boxplot by veg

Distribution of `gvf_vs_ndvi_gap` by vegetation type for **every** scores CSV.
Spin-up sites are red diamonds so we can see how much they inflate the apparent
discrepancy (especially EN / GR; DB barely moves).

True lag / compression examples and the DB-vs-shrub/mixed effect-size test are
in the section after golden ranking (same clean, non-spin-up pool).

Writes `anomaly_pipeline/output/boxplot/<FOLDER>_BOXPLOT.png`.


In [16]:
boxplot_paths = []
for csv_path in sorted(METADATA_DIR.glob("*_scores.csv")):
    out = plot_gap_boxplot_by_veg(csv_path, ANOMALY_DIR)
    boxplot_paths.append(out)
    print(out)


Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2023_BOXPLOT.png
/Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2023_BOXPLOT.png
Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2024_BOXPLOT.png
/Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GBOV_2024_BOXPLOT.png
Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GoldenSites_2023_BOXPLOT.png
/Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxplot/GoldenSites_2023_BOXPLOT.png
Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/boxpl

## Golden standard ranking

Drop spin-up, then rank sites by combined GVF-GCC / GVF-NDVI divergence (gap +
DTW). Closed-canopy **DB** sites are flagged as the control group: most uniform
at VIIRS scales, tightest cross-product agreement, almost no spin-up. Their
gap/DTW distribution is the irreducible baseline under ideal conditions.

Top ranks ≈ small disagreement (baseline); mid/lower ranks still include
larger offsets. The next section pulls lag/compression examples from metadata
and tests whether shrub/mixed gap exceeds the DB baseline (Cohen's d).

Needs scores under `output/metadata/`. Writes `output/golden_standard_ranking.csv`.


In [14]:
rank_path = build_golden_ranking(ANOMALY_DIR)
rank = load_table(rank_path)
rank_cols = [
    "rank", "site", "veg", "year", "source", "golden_candidate",
    "combined_div", "combined_gap", "combined_dtw",
]
rank_cols = [c for c in rank_cols if c in rank.columns]
print(rank_path, "|", len(rank), "rows |", int(rank["golden_candidate"].sum()), "DB candidates")
#overall combined score is the sum of gap, div, and dtw
display(rank.head(15)[rank_cols]) #top 15 overall best combined score in any veg
# display(rank.loc[rank["golden_candidate"]].head(15)[rank_cols]) #top 15 DB candidates


Ranked 59 site-years (27 DB golden candidates); excluded spin-up=True
Wrote /Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/golden_standard_ranking.csv
/Applications/Home/All School/School Past/2026 Summer/CISESS/NOAA-Phenology-Validation/anomaly_pipeline/output/golden_standard_ranking.csv | 59 rows | 27 DB candidates


,rank,site,veg,year,source,golden_candidate,combined_div,combined_gap,combined_dtw
0,1,blackrockforest,DB,2023,GoldenSites_2023,True,0.553277,7.250,0.035420
1,2,SRER,SH,2024,GBOV_2024,False,0.708393,9.125,0.056607
2,3,robinson2,DB,2023,GoldenSites_2023,True,0.919424,12.500,0.026567
3,4,HARV,DB,2024,GBOV_2024,True,1.019032,13.750,0.036889
4,5,bigtraillake,EN,2023,GoldenSites_2023,False,1.039068,13.750,0.056925
5,6,willowcreek,DB,2023,GoldenSites_2023,True,1.039389,13.625,0.066175
6,7,BART,DB,2024,GBOV_2024,True,1.126926,15.250,0.037641
7,8,morganmonroe2,DB,2023,GoldenSites_2023,True,1.143439,15.625,0.027367
8,9,willowcreek,DB,2024,GoldenSites_2024,True,1.181188,15.625,0.065117
9,10,dukehw,DB,2023,GoldenSites_2023,True,1.252109,17.125,0.028894


## Lag, compression, and effect size vs DB baseline

Same clean pool as ranking (spin-up excluded). Two related questions in one pass:

1. **Examples from the CSVs** spotting lag/compression in
   `metadata/` (and why they are not at the top of
   `golden_standard_ranking.csv`):
   - **Lag-ish** (per phase): `lag_sos` / `lag_mos` / `lag_dos` / `lag_eos`
     = `gvf_* − ndvi_*` (``lag`` still means SOS). Look for large |lag| with a
     plausible green-up span (not spin-up).
      - Positive: GVF after NDVI (GVF later)
      - 0: same DOY
      - Negative: GVF before NDVI (GVF earlier)
      - |lag| guide: ~0–15 typical | 15–40 worth a look | 40+ lag candidate.
   - **Compression-ish (`greenup_comp`):** green-up length ratio
     `(gvf_mos−gvf_sos)/(gcc_mos−gcc_sos)` 
      - ratio = 1 -> GVF's green-up phase took exactly as many days as GCC's. No stretching, no squeezing
      - ratio < 1 -> the numerator (GVF's duration) is smaller than the denominator (GCC's duration). GVF's green-up happened in fewer days than GCC's and GVF is compressed relative to GCC.
      - ratio > 1 ->  GVF's duration is bigger. GVF took longer to go from onset to peak than GCC did, GVF is stretched relative to GCC.
   - **Senescence** (`senescence_comp`): same idea for DOS→EOS, `(gvf_eos−gvf_dos)/(gcc_eos−gcc_dos)` (=1 same length, <1 GVF shorter/compressed, >1 GVF longer/stretched).

2. **Effect-size test** — is shrub/mixed (`SH`+`GR`+`EN`) `gvf_vs_ndvi_gap`
   larger than the DB golden-standard mean? Cohen's d + one-sided Welch t-test
   (`H1: mixed > DB`). Only *excess* beyond the DB baseline supports a
   land-cover-driven lag claim.

Caveat: small `n` for shrub/mixed means the test can be underpowered; treat a
non-significant result as "not yet demonstrated," not proof of no effect.


### CODE: Shared helpers

Load all scores, drop spin-up, and define lollipop helpers + Cohen's d used by
the GoldenSites / GBOV cells below.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import Image, display
from scipy import stats

# --- shared clean pool (used by GoldenSites + GBOV cells) ---
all_scores = load_all_scores(ANOMALY_DIR)
all_scores["spin_up"] = all_scores["gvf_sos"].eq(1.0)
clean = all_scores.loc[~all_scores["spin_up"]].copy()
print(
    f"rows={len(all_scores)} | spin-up={int(all_scores['spin_up'].sum())} | "
    f"clean={len(clean)} | sources={sorted(all_scores['source'].unique())}"
)

show_cols = [
    c for c in [
        "site", "lag", "lag_sos", "lag_mos", "lag_dos", "lag_eos",
        "greenup_comp", "senescence_comp", "veg", "year", "source",
        "gvf_sos", "ndvi_sos", "gvf_mos", "ndvi_mos",
        "gvf_dos", "ndvi_dos", "gvf_eos", "ndvi_eos",
        "gcc_sos", "gcc_mos", "gcc_dos", "gcc_eos",
        "gvf_vs_ndvi_gap", "gvf_vs_ndvi_dtw",
    ] if c in clean.columns
]


def _lollipop(ax, labels, values, color, ref_line=None):
    """Horizontal lollipop chart (cleaner than thick bars for ranked site values)."""
    y = np.arange(len(labels))
    vals = np.asarray(values, dtype=float)
    ax.hlines(y, 0 if ref_line is None else ref_line, vals, color=color, alpha=0.55, linewidth=1.4)
    ax.scatter(vals, y, color=color, s=42, zorder=3, edgecolors="white", linewidths=0.4)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=7)
    if ref_line is not None:
        ax.axvline(ref_line, color="black", linestyle="--", linewidth=1, alpha=0.75)
    ax.grid(True, axis="x", alpha=0.3)


def plot_lag_ranked_by_veg(df: pd.DataFrame, out_png: Path, title_prefix: str, show: bool = True):
    """Lag by phase (SOS/MOS/DOS/EOS) × veg as lollipops.

    Rows = phases, columns = veg types. Phase label sits at the top-right of each row.
    lag_* = gvf_* − ndvi_* (positive => GVF later).
    """
    phases = [
        ("lag_sos", "SOS"),
        ("lag_mos", "MOS"),
        ("lag_dos", "DOS"),
        ("lag_eos", "EOS"),
    ]
    plot_df = df.copy()
    # tolerate older frames that only have ``lag``
    if "lag_sos" not in plot_df.columns and "lag" in plot_df.columns:
        plot_df["lag_sos"] = plot_df["lag"]

    vegs = sorted(plot_df["veg"].dropna().unique())
    n_veg = max(len(vegs), 1)
    max_n = 4
    for col, _ in phases:
        if col in plot_df.columns and plot_df[col].notna().any():
            max_n = max(max_n, int(plot_df.dropna(subset=[col]).groupby("veg").size().max()))

    fig, axes = plt.subplots(
        len(phases),
        n_veg,
        figsize=(5.8 * n_veg, max(3.4, 0.32 * max_n + 1.6) * len(phases)),
        sharex=False,
        squeeze=False,
    )
    colors = plt.cm.tab10(np.linspace(0, 1, max(n_veg, 1)))

    for row_i, (col, phase) in enumerate(phases):
        for col_i, (veg, color) in enumerate(zip(vegs, colors)):
            ax = axes[row_i][col_i]
            if col not in plot_df.columns:
                ax.set_visible(False)
                continue
            sub = (
                plot_df.loc[plot_df["veg"].eq(veg)]
                .dropna(subset=[col, "site"])
                .sort_values(col, ascending=True)
            )
            if sub.empty:
                ax.set_visible(False)
                continue
            _lollipop(ax, list(sub["site"]), sub[col].values, color, ref_line=0)
            ax.tick_params(axis="y", pad=6)
            if row_i == 0:
                ax.set_title(f"{veg} (n={len(sub)})")
            ax.set_xlabel(f"{phase} lag (days)")

        # phase label in the right margin (top of each row), clear of the panels
        right_ax = axes[row_i][-1]
        right_ax.text(
            1.28,
            0.95,
            phase,
            transform=right_ax.transAxes,
            ha="left",
            va="top",
            fontsize=12,
            fontweight="bold",
            clip_on=False,
        )

    fig.suptitle(f"{title_prefix} — Phase lag (GVF − NDVI) by veg", y=0.995)
    # large bottom/right margins: legend below EOS, phase tags clear of last column
    fig.subplots_adjust(left=0.08, right=0.90, top=0.93, bottom=0.22, hspace=0.75, wspace=1.15)
    fig.legend(
        handles=[
            Line2D([0], [0], color="none", label="lag_sos/mos/dos/eos = gvf_* − ndvi_*"),
            Line2D([0], [0], color="none", label="+ : GVF after NDVI (GVF later)"),
            Line2D([0], [0], color="none", label="0 : same DOY"),
            Line2D([0], [0], color="none", label="− : GVF before NDVI (GVF earlier)"),
            Line2D([0], [0], color="none", label="|lag| guide: ~0–15 typical | 15–40 look | 40+ candidate"),
        ],
        loc="upper center",
        bbox_to_anchor=(0.47, 0.18),
        ncol=1,
        frameon=True,
        fontsize=8,
    )
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight", pad_inches=0.5)
    plt.close(fig)
    if show:
        display(Image(filename=str(out_png)))
    print(f"Wrote {out_png}")


def plot_compression_ranked_by_veg(df: pd.DataFrame, out_png: Path, title_prefix: str, show: bool = True):
    """greenup_comp | senescence_comp as separate side-by-side lollipop panels per veg."""
    gu_color = "#4C78A8"
    sen_color = "#F58518"
    metrics = [
        ("greenup_comp", gu_color, "greenup_comp = (gvf_mos−gvf_sos)/(gcc_mos−gcc_sos)"),
        ("senescence_comp", sen_color, "senescence_comp = (gvf_eos−gvf_dos)/(gcc_eos−gcc_dos)"),
    ]
    vegs = sorted(df["veg"].dropna().unique())
    n_veg = max(len(vegs), 1)
    max_n = 4
    for col, _, _ in metrics:
        if col in df.columns and df[col].notna().any():
            max_n = max(max_n, int(df.dropna(subset=[col]).groupby("veg").size().max()))

    fig, axes = plt.subplots(
        n_veg, 2,
        figsize=(13, max(3.8, 0.40 * max_n + 1.8) * n_veg),
        sharex=False,
        squeeze=False,
    )
    for row, veg in enumerate(vegs):
        for col_i, (col, color, _) in enumerate(metrics):
            ax = axes[row][col_i]
            sub = df.loc[df["veg"].eq(veg)].dropna(subset=[col, "site"]).sort_values(col, ascending=True)
            if sub.empty:
                ax.set_visible(False)
                continue
            _lollipop(ax, list(sub["site"]), sub[col].values, color, ref_line=1)
            ax.set_xlabel(col)
            if col_i == 0:
                ax.set_ylabel(veg)
            ax.set_title(f"{veg} · {col} (n={len(sub)})")

    fig.suptitle(
        f"{title_prefix} — greenup_comp (left) | senescence_comp (right)",
        y=1.01,
    )
    fig.legend(
        handles=[
            Line2D([0], [0], color=gu_color, lw=6, label=metrics[0][2]),
            Line2D([0], [0], color=sen_color, lw=6, label=metrics[1][2]),
            Line2D([0], [0], color="none", label="ratio = 1 : same length as GCC"),
            Line2D([0], [0], color="none", label="ratio < 1 : GVF shorter (compressed vs GCC)"),
            Line2D([0], [0], color="none", label="ratio > 1 : GVF longer (stretched vs GCC)"),
        ],
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=1,
        frameon=True,
        fontsize=8,
    )
    fig.tight_layout(rect=[0.0, 0.08, 1.0, 0.97])
    fig.subplots_adjust(hspace=0.85, wspace=0.45)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight", pad_inches=0.4)
    plt.close(fig)
    if show:
        display(Image(filename=str(out_png)))
    print(f"Wrote {out_png}")


def cohens_d(a: pd.Series, b: pd.Series) -> float:
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return float("nan")
    var_p = ((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2)
    return (b.mean() - a.mean()) / np.sqrt(var_p)


### GoldenSites 2023

Tables with `lag` / `greenup_comp` / `senescence_comp` for **GoldenSites_2023**
(spin-up excluded). Lollipop PNGs are saved under `output/lollipopPlot/` (not shown here).


In [ ]:
gs = clean.loc[clean["source"].eq("GoldenSites_2023")].copy()
print(f"GoldenSites_2023 clean n={len(gs)}:")
if gs.empty:
    print("No clean rows; skip.")
else:
    display(gs[show_cols])

    lag_png = ANOMALY_DIR / "lollipopPlot" / "2023_GoldenSite_Lag.png"
    comp_png = ANOMALY_DIR / "lollipopPlot" / "2023_GoldenSite_Compression.png"
    plot_lag_ranked_by_veg(gs, lag_png, "2023 GoldenSites", show=False)
    plot_compression_ranked_by_veg(gs, comp_png, "2023 GoldenSites", show=False)

    # effect size on full clean pool (DB vs shrub/mixed across sources)
    db_gap = clean.loc[clean["veg"].eq("DB"), "gvf_vs_ndvi_gap"].dropna()
    mixed_gap = clean.loc[clean["veg"].isin(["SH", "GR", "EN"]), "gvf_vs_ndvi_gap"].dropna()

    d = cohens_d(db_gap, mixed_gap)
    tt = stats.ttest_ind(mixed_gap, db_gap, equal_var=False, alternative="greater")

    print("\nEffect size: shrub/mixed (SH+GR+EN) vs DB golden baseline (gvf_vs_ndvi_gap)")
    print(f"  DB mean gap:      {db_gap.mean():.1f} days (n={len(db_gap)})")
    print(f"  Shrub/mixed mean: {mixed_gap.mean():.1f} days (n={len(mixed_gap)})")
    print(f"  Cohen's d:        {d:.2f}  (|d|<0.2 negligible, ~0.5 medium, ~0.8 large)")
    print(f"  one-sided p:      {tt.pvalue:.3f}  (H1: mixed > DB)")
    if tt.pvalue >= 0.05:
        print(
            "  -> not significant: after dropping spin-up, shrub/mixed does not show a "
            "detectable excess lag over DB (underpowered if n_mixed is small)."
        )
    else:
        print(
            "  -> significant excess gap in shrub/mixed beyond the DB baseline "
            "(still check n and spin-up screening before claiming ecology)."
        )


### GoldenSites 2024

Tables with `lag` / `greenup_comp` / `senescence_comp` for **GoldenSites_2024**
(spin-up excluded). Lollipop PNGs are saved under `output/lollipopPlot/` (not shown here).

Lollipop plots omit veg codes **EN**, **WL**, and **UN**.


In [ ]:
gs24 = clean.loc[clean["source"].eq("GoldenSites_2024")].copy()
print(f"GoldenSites_2024 clean n={len(gs24)}:")
if gs24.empty:
    print("No clean (non-spin-up) rows for GoldenSites_2024; skip plots/tests.")
else:
    display(gs24[show_cols])

    # omit WL / UN from lollipop visualizations only
    gs24_plot = gs24.loc[~gs24["veg"].isin(["EN", "WL", "UN"])].copy()
    print(f"  plot n={len(gs24_plot)} after dropping EN/WL/UN")

    lag_png = ANOMALY_DIR / "lollipopPlot" / "2024_GoldenSite_Lag.png"
    comp_png = ANOMALY_DIR / "lollipopPlot" / "2024_GoldenSite_Compression.png"
    plot_lag_ranked_by_veg(gs24_plot, lag_png, "2024 GoldenSites", show=False)
    plot_compression_ranked_by_veg(gs24_plot, comp_png, "2024 GoldenSites", show=False)

    # effect size within GoldenSites_2024 only
    db_gap = gs24.loc[gs24["veg"].eq("DB"), "gvf_vs_ndvi_gap"].dropna()
    mixed_gap = gs24.loc[gs24["veg"].isin(["SH", "GR", "EN"]), "gvf_vs_ndvi_gap"].dropna()

    d = cohens_d(db_gap, mixed_gap)
    tt = stats.ttest_ind(mixed_gap, db_gap, equal_var=False, alternative="greater")

    print("\nEffect size (GoldenSites_2024 only): shrub/mixed (SH+GR+EN) vs DB (gvf_vs_ndvi_gap)")
    print(f"  DB mean gap:      {db_gap.mean():.1f} days (n={len(db_gap)})")
    print(f"  Shrub/mixed mean: {mixed_gap.mean():.1f} days (n={len(mixed_gap)})")
    print(f"  Cohen's d:        {d:.2f}  (|d|<0.2 negligible, ~0.5 medium, ~0.8 large)")
    print(f"  one-sided p:      {tt.pvalue:.3f}  (H1: mixed > DB)")
    if len(db_gap) < 2 or len(mixed_gap) < 2:
        print("  -> too few sites in one group for a stable test; treat descriptively.")
    elif tt.pvalue >= 0.05:
        print(
            "  -> not significant: after dropping spin-up, shrub/mixed does not show a "
            "detectable excess lag over DB (underpowered if n_mixed is small)."
        )
    else:
        print(
            "  -> significant excess gap in shrub/mixed beyond the DB baseline "
            "(still check n and spin-up screening before claiming ecology)."
        )


### GBOV 2023 and GBOV 2024

Tables with `lag` / `greenup_comp` / `senescence_comp` for **GBOV_2023** and
**GBOV_2024** (spin-up excluded). Lollipop PNGs are saved under `output/lollipopPlot/`
(not shown here).


In [ ]:
for source, file_tag, title in [
    ("GBOV_2023", "2023_GBOV", "2023 GBOV"),
    ("GBOV_2024", "2024_GBOV", "2024 GBOV"),
]:
    subset = clean.loc[clean["source"].eq(source)].copy()
    print(f"\n=== {source} clean n={len(subset)} ===")
    if subset.empty:
        print(f"No clean rows for {source}; skip.")
        continue

    display(subset[show_cols])

    lag_png = ANOMALY_DIR / "lollipopPlot" / f"{file_tag}_Lag.png"
    comp_png = ANOMALY_DIR / "lollipopPlot" / f"{file_tag}_Compression.png"
    plot_lag_ranked_by_veg(subset, lag_png, title, show=False)
    plot_compression_ranked_by_veg(subset, comp_png, title, show=False)


### Combined lag: GBOV + GoldenSites (2023 vs 2024)

One figure for **AG / DB / GR / SH** only. Rows = SOS / MOS / DOS / EOS.
If the same site appears in both years, draw two lollipops close together
(**2023 = blue**, **2024 = green**); different sites are spaced farther apart.
X-axis is always centered at 0 (symmetric limits from the panel max |lag|).


In [ ]:
# Combined lag across GBOV + GoldenSites; pair 2023/2024 for the same site
VEG_KEEP = ["AG", "DB", "GR", "SH"]
SOURCES = [
    "GBOV_2023", "GBOV_2024",
    "GoldenSites_2023", "GoldenSites_2024",
]
YEAR_COLOR = {2023: "#1f77b4", 2024: "#2ca02c"}  # blue / green
PHASES = [
    ("lag_sos", "SOS"),
    ("lag_mos", "MOS"),
    ("lag_dos", "DOS"),
    ("lag_eos", "EOS"),
]

# within-site pair gap (small) vs between-site gap (larger)
PAIR_HALF = 0.20
SITE_STEP = 1.55


def plot_combined_lag_years(
    df: pd.DataFrame,
    out_png: Path,
    title_prefix: str = "GBOV + GoldenSites",
    show: bool = False,
):
    """Phase × veg lag lollipops; 2023/2024 paired for the same site."""
    plot_df = df.copy()
    if "lag_sos" not in plot_df.columns and "lag" in plot_df.columns:
        plot_df["lag_sos"] = plot_df["lag"]
    if "year" not in plot_df.columns:
        plot_df["year"] = plot_df["source"].str.extract(r"(20\d{2})")[0].astype(float)

    vegs = [v for v in VEG_KEEP if v in set(plot_df["veg"].dropna())]
    n_veg = max(len(vegs), 1)

    # precompute site order + y layout per veg (shared across phase rows)
    layout = {}
    max_sites = 1
    for veg in vegs:
        sub = plot_df.loc[plot_df["veg"].eq(veg)]
        sites = sorted(sub["site"].dropna().unique())
        max_sites = max(max_sites, len(sites))
        y_centers = []
        y_tick_pos = []
        y_tick_lab = []
        y = 0.0
        for site in sites:
            yrs = sorted({int(y) for y in sub.loc[sub["site"].eq(site), "year"].dropna()})
            y_centers.append((site, yrs, y))
            y_tick_pos.append(y)
            y_tick_lab.append(site)
            y += SITE_STEP
        layout[veg] = {
            "centers": y_centers,
            "ticks": y_tick_pos,
            "labels": y_tick_lab,
            "ymax": max(y - SITE_STEP, 0.0),
        }

    fig, axes = plt.subplots(
        len(PHASES),
        n_veg,
        figsize=(5.8 * n_veg, max(4.0, 0.48 * max_sites + 2.0) * len(PHASES)),
        sharex=False,
        squeeze=False,
    )

    for row_i, (col, phase) in enumerate(PHASES):
        # first pass: gather values so every panel in this row shares a symmetric x-limit
        row_vals = []
        panel_artists = []  # (ax, list of (yy, val, year))
        for col_i, veg in enumerate(vegs):
            ax = axes[row_i][col_i]
            info = layout[veg]
            sub = plot_df.loc[plot_df["veg"].eq(veg)].copy()
            if "year" in sub.columns:
                sub["_year_i"] = pd.to_numeric(sub["year"], errors="coerce").astype("Int64")
            else:
                sub["_year_i"] = pd.NA

            drawn = []
            for site, yrs, y0 in info["centers"]:
                site_rows = sub.loc[sub["site"].eq(site)]
                if 2023 in yrs and 2024 in yrs:
                    year_y = {2023: y0 - PAIR_HALF, 2024: y0 + PAIR_HALF}
                elif yrs:
                    year_y = {yrs[0]: y0}
                else:
                    year_y = {}

                for year, yy in year_y.items():
                    r = site_rows.loc[site_rows["_year_i"].eq(year)]
                    if r.empty or col not in r.columns:
                        continue
                    val = r.iloc[0][col]
                    if pd.isna(val):
                        continue
                    val = float(val)
                    drawn.append((yy, val, year))
                    row_vals.append(val)

            panel_artists.append((ax, info, drawn))

        if row_vals:
            lim = max(abs(v) for v in row_vals)
            lim = max(lim * 1.10, 5.0)
        else:
            lim = 5.0

        for col_i, (ax, info, drawn) in enumerate(panel_artists):
            for yy, val, year in drawn:
                color = YEAR_COLOR.get(year, "#555555")
                ax.hlines(yy, 0, val, color=color, alpha=0.65, linewidth=1.6)
                ax.scatter(
                    [val], [yy], color=color, s=40, zorder=3,
                    edgecolors="white", linewidths=0.4,
                )

            ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.8)
            ax.set_yticks(info["ticks"])
            ax.set_yticklabels(info["labels"], fontsize=7)
            ax.set_ylim(-0.55, info["ymax"] + 0.55)
            ax.invert_yaxis()
            ax.grid(True, axis="x", alpha=0.3)
            # force 0 at the visual center; stems scale left/right within +/- lim
            ax.set_xlim(-lim, lim)
            ax.set_autoscalex_on(False)

            if row_i == 0:
                ax.set_title(f"{vegs[col_i]} (n_sites={len(info['labels'])})")
            ax.set_xlabel(f"{phase} lag (days)")

        right_ax = axes[row_i][-1]
        right_ax.text(
            1.20, 0.95, phase,
            transform=right_ax.transAxes,
            ha="left", va="top", fontsize=12, fontweight="bold", clip_on=False,
        )

    fig.suptitle(
        f"{title_prefix} — Phase lag 2023 (blue) vs 2024 (green)",
        y=0.995,
    )
    fig.subplots_adjust(left=0.09, right=0.90, top=0.93, bottom=0.20, hspace=0.80, wspace=1.05)
    fig.legend(
        handles=[
            Line2D([0], [0], color=YEAR_COLOR[2023], lw=3, marker="o", label="2023"),
            Line2D([0], [0], color=YEAR_COLOR[2024], lw=3, marker="o", label="2024"),
            Line2D([0], [0], color="none", label="same site: paired lines close together"),
            Line2D([0], [0], color="none", label="lag_* = gvf_* − ndvi_*  |  0 centered on each panel"),
            Line2D([0], [0], color="none", label="+ later GVF | − earlier GVF"),
        ],
        loc="upper center",
        bbox_to_anchor=(0.47, 0.16),
        ncol=1,
        frameon=True,
        fontsize=8,
    )
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight", pad_inches=0.45)
    plt.close(fig)
    if show:
        display(Image(filename=str(out_png)))
    print(f"Wrote {out_png}")


combined = clean.loc[
    clean["source"].isin(SOURCES) & clean["veg"].isin(VEG_KEEP)
].copy()
# label GBOV sites with a GBOV_ prefix (GoldenSites keep bare names)
combined["site_label"] = combined["site"].astype(str)
gbov_mask = combined["source"].str.startswith("GBOV_")
combined.loc[gbov_mask, "site_label"] = "GBOV_" + combined.loc[gbov_mask, "site_label"]
print(
    f"combined pool n={len(combined)} | sites={combined['site_label'].nunique()} | "
    f"veg={sorted(combined['veg'].unique())}"
)

out = ANOMALY_DIR / "lollipopPlot" / "Combined_2023_2024_Lag.png"
# plot uses site_label when present
_plot_df = combined.copy()
if "site_label" in _plot_df.columns:
    _plot_df["site"] = _plot_df["site_label"]
plot_combined_lag_years(_plot_df, out, show=False)


### Combined compression: GBOV + GoldenSites (2023 vs 2024)

Same pool/rules as combined lag (**AG / DB / GR / SH**; `GBOV_` prefix;
**2023 = blue**, **2024 = green**; same-site pairs close together).

Axis is centered at **0** (0 = same length as GCC, + stretched, − compressed).
Rows = greenup_comp / senescence_comp.


In [ ]:
# Combined compression (ratio-1), 2023 blue / 2024 green, 0-centered
# (self-contained: rebuilds the same AG/DB/GR/SH pool as the combined-lag cell)
VEG_KEEP = ["AG", "DB", "GR", "SH"]
SOURCES = [
    "GBOV_2023", "GBOV_2024",
    "GoldenSites_2023", "GoldenSites_2024",
]
YEAR_COLOR = {2023: "#1f77b4", 2024: "#2ca02c"}
PAIR_HALF = 0.20
SITE_STEP = 1.55

combined = clean.loc[
    clean["source"].isin(SOURCES) & clean["veg"].isin(VEG_KEEP)
].copy()
combined["site_label"] = combined["site"].astype(str)
gbov_mask = combined["source"].str.startswith("GBOV_")
combined.loc[gbov_mask, "site_label"] = "GBOV_" + combined.loc[gbov_mask, "site_label"]
_plot_df = combined.copy()
_plot_df["site"] = _plot_df["site_label"]
print(
    f"combined compression pool n={len(_plot_df)} | sites={_plot_df['site'].nunique()} | "
    f"veg={sorted(_plot_df['veg'].unique())}"
)

COMP_METRICS = [
    ("greenup_comp", "greenup_comp", "greenup = (gvf_mos−gvf_sos)/(gcc_mos−gcc_sos)"),
    ("senescence_comp", "senescence_comp", "senescence = (gvf_eos−gvf_dos)/(gcc_eos−gcc_dos)"),
]


def plot_combined_compression_years(
    df: pd.DataFrame,
    out_png: Path,
    title_prefix: str = "GBOV + GoldenSites",
    show: bool = False,
):
    """Compression × veg lollipops; plot (ratio-1) so 0 is centered."""
    plot_df = df.copy()
    if "year" not in plot_df.columns:
        plot_df["year"] = plot_df["source"].str.extract(r"(20\d{2})")[0].astype(float)

    vegs = [v for v in VEG_KEEP if v in set(plot_df["veg"].dropna())]
    n_veg = max(len(vegs), 1)

    layout = {}
    max_sites = 1
    for veg in vegs:
        sub = plot_df.loc[plot_df["veg"].eq(veg)]
        sites = sorted(sub["site"].dropna().unique())
        max_sites = max(max_sites, len(sites))
        y_centers = []
        y_tick_pos = []
        y_tick_lab = []
        y = 0.0
        for site in sites:
            yrs = sorted({int(y) for y in sub.loc[sub["site"].eq(site), "year"].dropna()})
            y_centers.append((site, yrs, y))
            y_tick_pos.append(y)
            y_tick_lab.append(site)
            y += SITE_STEP
        layout[veg] = {
            "centers": y_centers,
            "ticks": y_tick_pos,
            "labels": y_tick_lab,
            "ymax": max(y - SITE_STEP, 0.0),
        }

    fig, axes = plt.subplots(
        len(COMP_METRICS),
        n_veg,
        figsize=(5.8 * n_veg, max(4.0, 0.48 * max_sites + 2.0) * len(COMP_METRICS)),
        sharex=False,
        squeeze=False,
    )

    for row_i, (col, phase, _) in enumerate(COMP_METRICS):
        row_vals = []
        panel_artists = []
        for col_i, veg in enumerate(vegs):
            ax = axes[row_i][col_i]
            info = layout[veg]
            sub = plot_df.loc[plot_df["veg"].eq(veg)].copy()
            sub["_year_i"] = pd.to_numeric(sub["year"], errors="coerce").astype("Int64")

            drawn = []
            for site, yrs, y0 in info["centers"]:
                site_rows = sub.loc[sub["site"].eq(site)]
                if 2023 in yrs and 2024 in yrs:
                    year_y = {2023: y0 - PAIR_HALF, 2024: y0 + PAIR_HALF}
                elif yrs:
                    year_y = {yrs[0]: y0}
                else:
                    year_y = {}

                for year, yy in year_y.items():
                    r = site_rows.loc[site_rows["_year_i"].eq(year)]
                    if r.empty or col not in r.columns:
                        continue
                    val = r.iloc[0][col]
                    if pd.isna(val):
                        continue
                    # ratio-1 so 0 = match GCC; keeps 0 at center like lag
                    delta = float(val) - 1.0
                    drawn.append((yy, delta, year))
                    row_vals.append(delta)

            panel_artists.append((ax, info, drawn))

        if row_vals:
            lim = max(abs(v) for v in row_vals)
            lim = max(lim * 1.10, 0.5)
        else:
            lim = 0.5

        for col_i, (ax, info, drawn) in enumerate(panel_artists):
            for yy, delta, year in drawn:
                color = YEAR_COLOR.get(year, "#555555")
                ax.hlines(yy, 0, delta, color=color, alpha=0.65, linewidth=1.6)
                ax.scatter(
                    [delta], [yy], color=color, s=40, zorder=3,
                    edgecolors="white", linewidths=0.4,
                )

            ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.8)
            ax.set_yticks(info["ticks"])
            ax.set_yticklabels(info["labels"], fontsize=7)
            ax.set_ylim(-0.55, info["ymax"] + 0.55)
            ax.invert_yaxis()
            ax.grid(True, axis="x", alpha=0.3)
            ax.set_xlim(-lim, lim)
            ax.set_autoscalex_on(False)

            if row_i == 0:
                ax.set_title(f"{vegs[col_i]} (n_sites={len(info['labels'])})")
            ax.set_xlabel(f"{phase} (centered at match-GCC)")

        right_ax = axes[row_i][-1]
        right_ax.text(
            1.20, 0.95, phase,
            transform=right_ax.transAxes,
            ha="left", va="top", fontsize=11, fontweight="bold", clip_on=False,
        )

    fig.suptitle(
        f"{title_prefix} — Compression 2023 (blue) vs 2024 (green)",
        y=0.995,
    )
    fig.subplots_adjust(left=0.09, right=0.90, top=0.92, bottom=0.22, hspace=0.80, wspace=1.05)
    fig.legend(
        handles=[
            Line2D([0], [0], color=YEAR_COLOR[2023], lw=3, marker="o", label="2023"),
            Line2D([0], [0], color=YEAR_COLOR[2024], lw=3, marker="o", label="2024"),
            Line2D([0], [0], color="none", label="0 : same length as GCC"),
            Line2D([0], [0], color="none", label="+ : GVF longer (stretched) | − : GVF shorter (compressed)"),
            Line2D([0], [0], color="none", label=COMP_METRICS[0][2]),
            Line2D([0], [0], color="none", label=COMP_METRICS[1][2]),
        ],
        loc="upper center",
        bbox_to_anchor=(0.47, 0.18),
        ncol=1,
        frameon=True,
        fontsize=8,
    )
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight", pad_inches=0.45)
    plt.close(fig)
    if show:
        display(Image(filename=str(out_png)))
    print(f"Wrote {out_png}")


out_comp = ANOMALY_DIR / "lollipopPlot" / "Combined_2023_2024_Compression.png"
plot_combined_compression_years(_plot_df, out_comp, show=False)


## Findings notes

### GoldenSites 2023

Veg codes: DB = deciduous broadleaf, EN = evergreen needle, GR = grassland, AG = agriculture, SH = shrub, EB = evergreen broadleaf

1) Compression: 
    - For Agriculture lands (n=6), green-up compression value shows mostly value greater than 1 which indiate gvf being stretched relative to gcc and senescence compression value also shows mostly value greater than 1 which is also stretched.
    - For deciduous broadleaf (n=14), green-up compression value is all positive which mean stretched. for senescence compression value its half site being stretched and half site being compressed. (could also say 2-3 site is a almost exact match (no stretch or compress, value = 1))
    - For evergreen needle and Grassland (both n = 1) all of the green-up and senescence seem to say it's being stretched
2) Lag:
    - EN and GR lag is insignificant 
    - For Agriculture lands (n=6), the lag various between -50 day eariler vs 50 day later SOS
    - For deciduous broadleaf (n=14), most site is a few days eariler but a few site are 50 days later SOS
